# Hospital Patient Intake and Department Routing Assistant
## Autonomous Multi-Agent Coordination via Microsoft Semantic Kernel & ChromaDB RAG

This interactive demonstration notebook runs the real application modules and executes all six specialized agents:
1. **Intake Agent** (`validate_patient_input`, `create_patient_record`)
2. **Records Memory Agent** (`search_patient_history`, `recall_previous_routing`)
3. **Safety Gate Agent** (`detect_urgent_warning_signs`, `generate_urgent_care_advisory`)
4. **Triage Router Agent** (`classify_patient_concern`, `get_department_information`)
5. **Information Agent** (`retrieve_department_information`, `format_grounded_instructions`)
6. **Appointment Agent** (`get_available_appointment_slots`, `reserve_appointment_slot`)

**Mandatory Regulatory Notice**: This system is strictly an administrative intake assistant, **NOT** a medical diagnostic device. It never diagnoses diseases or prescribes medication.

### Step 1: Initialize Environment and Import Modules

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import semantic_kernel as sk
from backend.database import init_db
from backend.rag.retriever import rag_retriever
from backend.orchestration.orchestrator import orchestrator
from backend.agents import (
    IntakeAgent,
    RecordsMemoryAgent,
    SafetyGateAgent,
    TriageRouterAgent,
    InformationAgent,
    AppointmentAgent
)

print(f"Loaded Microsoft Semantic Kernel version: {sk.__version__}")
init_db()
print("SQLite database initialized.")

### Step 2: Index Hospital Policy Documents into ChromaDB Vector Store

In [ ]:
chunk_count = rag_retriever.load_and_index_documents(force_reload=False)
print(f"Active ChromaDB vector collection size: {chunk_count} grounded document chunks.")

### Step 3: Run Scenario 1 - Normal Patient Routing (Knee Pain)
Input: *"I have knee pain for three days after stepping awkwardly."*

In [ ]:
payload_scenario1 = {
    "patient_name": "Arthur Pendelton",
    "age": 48,
    "described_concern": "I have severe knee pain for three days with joint stiffness and difficulty walking.",
    "duration": "3 days",
    "additional_info": "No prior surgery",
    "is_returning_patient": False
}

res1 = orchestrator.process_intake(payload_scenario1)

print("=== SCENARIO 1 ROUTING RESULTS ===")
print(f"Session ID:          {res1['session_id']}")
print(f"Assigned Department: {res1['selected_department']} ({res1['department_code']})")
print(f"Confidence Score:    {int(res1['confidence_score'] * 100)}%")
print(f"Routing Reason:      {res1['routing_reason']}")
print(f"Safety Status:       {res1['safety_status']} (Urgent: {res1['is_urgent']})")
print(f"Slots Available:     {len(res1['available_appointment_slots'])}")
print(f"Source Attributions: {[s['document_name'] for s in res1['source_references']]}")

print("\n--- Agent Execution Timeline ---")
for trace in res1["agent_traces"]:
    print(f"[{trace['status']:<18}] {trace['agent_name']:<22} | Tools: {', '.join(trace['tools_executed'])}")

### Step 4: Run Scenario 2 - Emergency Safety Gate Halt
Input: *"I have severe chest pain and difficulty breathing with cold sweats."*

In [ ]:
payload_scenario2 = {
    "patient_name": "Sarah Connor",
    "age": 56,
    "described_concern": "I have severe chest pain and difficulty breathing with radiating arm pressure.",
    "duration": "45 minutes",
    "additional_info": "History of hypertension",
    "is_returning_patient": False
}

res2 = orchestrator.process_intake(payload_scenario2)

print("=== SCENARIO 2 EMERGENCY HALT RESULTS ===")
print(f"Safety Status:        {res2['safety_status']}")
print(f"Urgent Flag:          {res2['is_urgent']}")
print(f"Warning Signs Found:  {res2['urgent_warning_signs']}")
print(f"Appointments Offered: {len(res2['available_appointment_slots'])} (Strictly 0 for emergency)")
print(f"Advisory Message:\n{res2['urgent_care_advisory']}")

print("\n--- Agent Traces (Notice Downstream Agents Stopped) ---")
for trace in res2["agent_traces"]:
    print(f"[{trace['status']:<18}] {trace['agent_name']:<22} | {trace['output_summary']}")

### Step 5: Run Scenario 3 - Returning Patient Memory Recall
Patient **Eleanor Vance (P-1001)** previously visited Orthopaedics, now presenting with a new dermatological complaint.

In [ ]:
payload_scenario3 = {
    "patient_name": "Eleanor Vance",
    "patient_id": "P-1001",
    "age": 45,
    "described_concern": "Developing an intensely itchy red rash with blisters across both forearms.",
    "duration": "4 days",
    "additional_info": "Applied new laundry detergent",
    "is_returning_patient": True
}

res3 = orchestrator.process_intake(payload_scenario3)

print("=== SCENARIO 3 RETURNING PATIENT RESULTS ===")
ret_ctx = res3['returning_patient_context']
print(f"Patient ID:          {res3['patient_id']}")
print(f"Returning Status:    {ret_ctx['is_returning']}")
print(f"Previous Visit Dept: {ret_ctx['previous_department']} on {ret_ctx['previous_routing_date']}")
print(f"Previous Concern:    {ret_ctx['previous_concern']}")
print(f"Today's Routed Dept: {res3['selected_department']} ({res3['department_code']})")
print(f"Evaluation Note:     Independent assessment conducted without bias from historical visit.")

### Step 6: Inspect Append-Only Audit Log (`logs/agent_runs.jsonl`)

In [ ]:
import json
log_file = project_root / "logs" / "agent_runs.jsonl"

if log_file.exists():
    lines = log_file.read_text(encoding="utf-8").strip().split("\n")
    print(f"Total append-only tool execution entries in log: {len(lines)}")
    print("\nMost recent 3 tool runs:")
    for line in lines[-3:]:
        entry = json.loads(line)
        print(f"- [{entry['timestamp']}] Agent: {entry['agent_name']} | Tool: {entry['tool_name']}() | Success: {entry['is_success']}")
else:
    print("Log file not created yet.")